### First LLM + RAG Project

#### Step 1: Load Documents

In [4]:
from langchain_community.document_loaders import PyPDFLoader
loader = PyPDFLoader("llama2.pdf")
doc = loader.load()

**Enable Lazy Loading** : 

If the document is very large, lazy loading means you don’t have to load the entire content in memory at once.

In [5]:
pages = []
async for page in loader.alazy_load():
    pages.append(page)

#### Step 2: Chunking Documents

In [6]:
from langchain.text_splitter import CharacterTextSplitter
text_splitter = CharacterTextSplitter(separator="\n",chunk_size=1000,chunk_overlap=200)
docs = text_splitter.split_documents(doc)

#### Step 3: Embedding Documents

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings
embedding_model=HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

# Embedding Model feature representation = 384
len(embedding_model.embed_query("hello AI"))

384

#### Step 4: Creating a Vector Store

FAISS = Facebook AI Similarity Search Engine (FAISS)

In [8]:
import faiss
from langchain_community.vectorstores import FAISS

# InMemoryDocstore stores documents in memory for quick retrieval by ID.
from langchain_community.docstore.in_memory import InMemoryDocstore

# Import Index
index=faiss.IndexFlatL2(384) #384 is dimension of embedding model

In [9]:
# Defining FAISS
vector_store=FAISS(
    embedding_function=embedding_model,
    index=index,
    docstore=InMemoryDocstore(), #stores documents in memory for quick retrieval by ID
    index_to_docstore_id={}, #maps index positions to document IDs
)

In [10]:
# add documnets to vector db
vector_store.add_documents(documents=docs)

['471a33ef-012b-405d-9599-8dd2777d8732',
 '5d7dcbc0-c736-4da9-bb18-2b6cd2e64104',
 '8a2f4fd3-425d-4ed2-b677-a76e41602897',
 'f41cc768-bfaf-4b40-a232-433d8c4e91d7',
 '163ba4d1-18fa-4942-90c0-e1245bd05e1c',
 'c413759b-1dac-4915-921f-9899978bf595',
 'bac856bd-5b94-458b-8daa-a9ae6e0959be',
 'ea516275-d847-4347-8aa9-03fb5d6d5ea4',
 '4f8b8353-d1d9-46ba-8fb6-b718249408c3',
 '4025ea74-d1cc-48a2-9d56-6978f1b72d8d',
 'b36965a7-b160-40ef-92ab-a7ca4a8b316e',
 'd4bf5dfe-626b-428e-aa5c-aa185a1bbc66',
 'a495ea18-2c80-451f-8b01-66b2fa21fe3e',
 'c22b0f94-4530-4e6c-af15-865e7db4a8c4',
 '68a05c9e-d259-42ca-9f6b-20b0b3625945',
 '3b15e5a9-4076-4d53-bd3d-3c2117c1ca7e',
 'ac502b5b-09d7-4a77-b30e-9fab322aea9e',
 'e42ec588-05ab-462f-8338-13e63a66f6a0',
 '74bad492-c1d4-41ae-bb22-8bb9482320c8',
 'c6e17320-1011-4aaa-be46-a894ade3a8dd',
 'd7ad4e80-bd43-4e06-9a9c-6c056a1ca1d1',
 'f7785b38-442c-448c-bc4f-f00c50db8ebb',
 '32265093-cf2c-47b7-bab0-670487ee319c',
 '1d5031e6-f1ad-4ba3-93af-8189e1d3a956',
 '0470a78f-48cf-

#### Checks 

- Similarity
- Filters

In [11]:
# Give Top 2 to my Query
vector_store.similarity_search(
    "what is llama2",
    k=2 #hyperparameter
)

[Document(id='f16dbbb9-554a-4c57-ac05-7e5365f64ea6', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'llama2.pdf', 'total_pages': 77, 'page': 30, 'page_label': '31'}, page_content='manceissimilaracrosscategories, Llama 2-Chathasrelativelymoreviolationsunderthe unqualifiedadvice\ncategory (although still low in an absolute sense), for various reasons, including lack of an appropriate\ndisclaimer (e.g.,“I am not a professional”) at times. For the other two categories,Llama 2-Chatachieves\ncomparable or lower violation percentage consistently regardless of model sizes.\nTruthfulness, Toxicity, and Bias.In Table 14, fine-tunedLlama 2-Chatshows great improvement over\nthe pretrainedLlama

In [12]:
# Adding Filters 
vector_store.similarity_search(
    "LangChain provides abstractions to make working with LLMs easy",
    #k=2 #hyperparameter,
    filter={"creator":{"$eq": "LaTeX with hyperref"}} 
)

[Document(id='8a2f4fd3-425d-4ed2-b677-a76e41602897', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'llama2.pdf', 'total_pages': 77, 'page': 0, 'page_label': '1'}, page_content='contribute to the responsible development of LLMs.\n∗Equal contribution, corresponding authors: {tscialom, htouvron}@meta.com\n†Second author\nContributions for all the authors can be found in Section A.1.\narXiv:2307.09288v2  [cs.CL]  19 Jul 2023'),
 Document(id='b97f3268-60cb-46a8-8083-bcab615ea763', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 

#### Step 5: Set Up Retreival

In [13]:
retriever=vector_store.as_retriever(
    search_kwargs={"k": 10} #hyperparameter
)

retriever.invoke("what is llama model?")

[Document(id='7c60f49f-1ef4-4d28-9aaa-fe99fb685df0', metadata={'producer': 'pdfTeX-1.40.25', 'creator': 'LaTeX with hyperref', 'creationdate': '2023-07-20T00:30:36+00:00', 'author': '', 'keywords': '', 'moddate': '2023-07-20T00:30:36+00:00', 'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5', 'subject': '', 'title': '', 'trapped': '/False', 'source': 'llama2.pdf', 'total_pages': 77, 'page': 48, 'page_label': '49'}, page_content='Llama 1\n7B 76.5 79.8 48.9 76.1 70.1 72.8 47.6 57.2 33.6 35.1\n13B 78.1 80.1 50.4 79.2 73.0 74.8 52.7 56.4 62.0 46.9\n33B 83.1 82.3 50.4 82.8 76.0 80.0 57.8 58.6 72.5 57.8\n65B 85.3 82.8 52.3 84.2 77.0 78.9 56.0 60.2 74.0 63.4\nLlama 2\n7B 77.4 78.8 48.3 77.2 69.2 75.2 45.9 58.6 57.8 45.3\n13B 81.7 80.5 50.3 80.7 72.8 77.3 49.4 57.0 67.3 54.8\n34B 83.7 81.9 50.9 83.3 76.7 79.4 54.5 58.2 74.3 62.6\n70B 85.0 82.8 50.7 85.3 80.2 80.2 57.4 60.2 78.5 68.9\nTable 20: Performance on standard benchmarks.\nHuman-E

#### Step 6: Generation

For Generation we need to set up RAG chain 

- Context - (Retriever)
- PromptTemplate - Langchain Hub
- Model - LLM
- OutPut Parser - StrOutputParser

In [14]:
# Model
from langchain_openai import ChatOpenAI
model=ChatOpenAI(temperature=7)

In [15]:
# Prompt Template from Langchain Hub
from langchain import hub
prompt = hub.pull("rlm/rag-prompt")

[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template="You are an assistant for question-answering tasks. Use the following pieces of retrieved context to answer the question. If you don't know the answer, just say that you don't know. Use three sentences maximum and keep the answer concise.\nQuestion: {question} \nContext: {context} \nAnswer:"), additional_kwargs={})]

In [16]:
# Output Parser
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

In [17]:
# Setup Format Docs to get page content from documents
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

#### RAG Chain

In [18]:
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

In [19]:
rag_chain.invoke("what is llama model?")

BadRequestError: Error code: 400 - {'error': {'message': "Invalid 'temperature': decimal above maximum value. Expected a value <= 2, but got 7.0 instead.", 'type': 'invalid_request_error', 'param': 'temperature', 'code': 'decimal_above_max_value'}}